# Odds Ratio Test: multiple testing and threshold diagnostics

Diagnostic plots for the stage-04 loss/duplication permulation test. They show how many per-HOG hits
survive multiple-testing correction and how the p-values behave. They also show how the occupancy
bounds and the permulation-derived LOR thresholds shape the hit lists.

1. Raw vs. BH vs. Bonferroni hit counts (with PhyloGLM for reference)
2. p-value histograms (expected uniform under the null; p = 0 and the 1/reps floor)
3. Occupancy range of each test and hit rate by occupancy
4. LOR vs. p-value plots with the two-sided and one-sided permulation thresholds

Reads `results_all.csv` and `results_summary.txt` from one results directory. It does not load the
large `results.pkl`.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

# Resolve repo root whether cwd is repo root, scripts/, or a stage subdirectory.
cwd = os.getcwd()
if os.path.basename(cwd) == "orb-selection":
    repo_root = cwd
elif os.path.basename(os.path.dirname(cwd)) == "orb-selection":
    repo_root = os.path.dirname(cwd)
elif os.path.basename(os.path.dirname(os.path.dirname(cwd))) == "orb-selection":
    repo_root = os.path.dirname(os.path.dirname(cwd))
else:
    repo_root = cwd

ort_dir = os.path.join(repo_root, "results/odds_ratio_test/Results_Aug12/Run1_occ_30-88_10000x")
phyloglm_csv = os.path.join(repo_root, "results/phyloglm/phyloglm.csv")

# Set to a directory (e.g. figures/supplementary) to save PNGs; None just displays them.
SAVE_DIR = os.environ.get("ORT_FIG_DIR")

ALPHA = 0.05  # two-sided level; each direction is called at ALPHA / 2
TAIL_ALPHA = ALPHA / 2
MIN_OCC = 30
MAX_OCC_LOSS = 88  # upper occupancy bound applies to loss only; duplication is unbounded on purpose

# Orb-weavers / non-orb-weavers colors match the other odds-ratio figures
COL = {"fg": "salmon", "bg": "steelblue"}
NAME = {"fg": "orb-weavers", "bg": "non-orb-weavers"}
INK, MUTED, GRID = "#222222", "#6b6b6b", "#e4e4e4"

plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.labelsize": 10, "figure.dpi": 110,
})


def save(fig, name):
    if SAVE_DIR:
        os.makedirs(SAVE_DIR, exist_ok=True)
        fig.savefig(os.path.join(SAVE_DIR, name), dpi=300, bbox_inches="tight")

## Load results and build a tidy table (one row per HOG per test)

In [ ]:
res = pd.read_csv(os.path.join(ort_dir, "results_all.csv"), index_col="HOG")
summary = open(os.path.join(ort_dir, "results_summary.txt")).read()

REPS = int(re.search(r"Permulation test with (\d+) repetitions", summary).group(1))


def grab(label):
    return float(re.search(rf"{label}: (-?\d+\.\d+)", summary).group(1))


# Averaged permulated distribution stats (3 d.p. in the summary), used for the LOR thresholds
perm_stats = {
    t: (grab(f"Permulated average mean, {w}"), grab(f"Permulated average standard deviation, {w}"))
    for t, w in (("loss", "loss"), ("dup", "duplication"))
}

TESTS = [("loss", "fg"), ("loss", "bg"), ("dup", "fg"), ("dup", "bg")]
WORD = {"loss": "loss", "dup": "duplication"}
LOR_COL = {"loss": "Log odds ratio of loss", "dup": "Log odds ratio of duplication"}

frames = []
for test, grp in TESTS:
    d = res[res["Occupancy"] >= MIN_OCC]
    if test == "loss":
        d = d[d["Occupancy"] <= MAX_OCC_LOSS]
    out = pd.DataFrame({
        "test": test, "group": grp, "occ": d["Occupancy"],
        "lor": d[LOR_COL[test]],
        "p": d[f"P-value {WORD[test]} more likely in {grp}"],
    })
    frames.append(out)
long = pd.concat(frames).reset_index()
# The fg and bg p-values of a test are two tails of one comparison: correct them together (stacked)
long["q_bh"] = np.nan
long["p_bonf"] = np.nan
for _, idx in long.groupby("test").groups.items():
    long.loc[idx, "q_bh"] = multipletests(long.loc[idx, "p"], method="fdr_bh")[1]
    long.loc[idx, "p_bonf"] = multipletests(long.loc[idx, "p"], method="bonferroni")[1]
long["label"] = long["test"].map(WORD) + ", " + long["group"].map(NAME)
print(f"{REPS} permulations -> smallest non-zero p = {1 / REPS:g}")
long.groupby(["test", "group"]).agg(
    tested=("p", "size"), p_zero=("p", lambda s: int((s == 0).sum())),
    raw=("p", lambda s: int((s <= TAIL_ALPHA).sum())),
    bh=("q_bh", lambda s: int((s < ALPHA).sum())),
    bonferroni=("p_bonf", lambda s: int((s < ALPHA).sum())),
)

## 1. Hits at raw p, BH and Bonferroni

PhyloGLM is the only analysis in the repo that already applies a correction (Bonferroni, in the PhyloGLM notebook).
It is shown here as a reference. Its test is two-sided, so it is not directly comparable to the one-sided odds-ratio tests.

In [ ]:
pg = pd.read_csv(phyloglm_csv, index_col="HOG")
pg_p = pd.to_numeric(pg["coef_orb_weavingTRUE_p.value"], errors="coerce").dropna()
pg_p = pg_p[np.isfinite(pg_p)]

counts = {}
for (test, grp), d in long.groupby(["test", "group"], sort=False):
    counts[f"{WORD[test]},\n{NAME[grp]}"] = (
        (d["p"] <= TAIL_ALPHA).sum(), (d["q_bh"] < ALPHA).sum(), (d["p_bonf"] < ALPHA).sum(), len(d))
counts["PhyloGLM\n(two-sided)"] = (
    (pg_p <= TAIL_ALPHA).sum(),
    (multipletests(pg_p, method="fdr_bh")[1] < ALPHA).sum(),
    (multipletests(pg_p, method="bonferroni")[1] < ALPHA).sum(), len(pg_p))

# One hue, light -> dark with correction strictness
SHADES = ["#c9dbea", "#7fa7c8", "#2c5d86"]
STEPS = ["Raw p \u2264 0.025", "BH q < 0.05 (tails stacked)", "Bonferroni < 0.05 (tails stacked)"]

fig, ax = plt.subplots(figsize=(10, 4.6))
xs = np.arange(len(counts))
w = 0.26
for k, (step, shade) in enumerate(zip(STEPS, SHADES)):
    vals = [v[k] for v in counts.values()]
    bars = ax.bar(xs + (k - 1) * w, vals, width=w - 0.03, color=shade, label=step)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 8, f"{v:,}", ha="center", va="bottom",
                fontsize=8.5, color=INK)
ax.set_xticks(xs)
ax.set_xticklabels([f"{k}\n(n={v[3]:,})" for k, v in counts.items()], fontsize=9)
ax.set_ylabel("Significant HOGs")
ax.set_title("Per-HOG hits before and after multiple-testing correction", loc="left")
ax.grid(axis="x", visible=False)
ax.set_ylim(0, max(v[0] for v in counts.values()) * 1.12)
ax.legend(frameon=False, fontsize=9, loc="upper left")
fig.tight_layout()
save(fig, "ort_hits_by_correction.png")
plt.show()

## 2. p-value distributions

If there were no real signal and the p-values were valid, each histogram would be flat (dashed line).
A spike near 0 means an excess of small p-values (signal). A tilt toward 1, or a total below the dashed line at p ≤ 0.025, means the null is conservative or miscalibrated.
The p = 0 HOGs are the only ones that survive correction, because the resolution is 1/reps.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10, 6.4), sharex=True)
nb = 20
for ax, (test, grp) in zip(axs.flat, TESTS):
    d = long[(long["test"] == test) & (long["group"] == grp)]
    ax.hist(d["p"], bins=np.linspace(0, 1, nb + 1), color=COL[grp], alpha=0.85,
            edgecolor="white", linewidth=0.8)
    ax.axhline(len(d) / nb, color=INK, linestyle="--", linewidth=1.1)
    n_le = int((d["p"] <= TAIL_ALPHA).sum())
    n0 = int((d["p"] == 0).sum())
    ax.set_title(f"{WORD[test].capitalize()} more likely in {NAME[grp]}", loc="left", pad=20)
    ax.text(0, 1.015,
            f"n = {len(d):,}  |  p \u2264 0.025: {n_le:,} ({n_le / len(d):.1%}; uniform null \u2248 2.5%)  |  p = 0: {n0}",
            transform=ax.transAxes, ha="left", va="bottom", fontsize=8.5, color=MUTED)
    ax.set_ylabel("HOGs")
    ax.grid(axis="x", visible=False)
for ax in axs[1]:
    ax.set_xlabel("Permulation p-value")
fig.legend(handles=[Line2D([], [], color=INK, linestyle="--", label="Expected count per bin if uniform")],
           frameon=False, fontsize=9, loc="lower center", bbox_to_anchor=(0.5, -0.01))
fig.suptitle(f"Permulation p-value histograms ({REPS:,} permulations; smallest non-zero p = {1 / REPS:g})",
             x=0.01, ha="left", fontsize=12)
fig.tight_layout(rect=(0, 0.03, 1, 0.96))
save(fig, "ort_pvalue_histograms.png")
plt.show()

## 3. Occupancy range and hit rate by occupancy

Loss is tested only at occupancy ≤ the upper bound so that "loss" is meaningful. Duplication has no upper bound, deliberately.
The right panel shows whether the hit rate depends on occupancy.

In [ ]:
occ_counts = res[res["Occupancy"] >= MIN_OCC]["Occupancy"].value_counts().sort_index()
max_occ_all = int(occ_counts.index.max())

fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 4.4), gridspec_kw={"width_ratios": [1.2, 1]})

axa.bar(occ_counts.index, occ_counts.values, width=0.85, color="#9aa6b2")
axa.axvspan(MIN_OCC - 0.5, MAX_OCC_LOSS + 0.5, ymin=0.9, ymax=1.0, color="#2c5d86", alpha=0.9)
axa.axvspan(MIN_OCC - 0.5, max_occ_all + 0.5, ymin=0.8, ymax=0.88, color="#7fa7c8", alpha=0.9)
ytop = occ_counts.max()
axa.text(MIN_OCC, ytop * 1.35 * 0.95, f"loss tested: occupancy {MIN_OCC}\u2013{MAX_OCC_LOSS}", color="white",
         fontsize=8.5, va="center")
axa.text(MIN_OCC, ytop * 1.35 * 0.84, f"duplication tested: occupancy \u2265 {MIN_OCC}", color=INK,
         fontsize=8.5, va="center")
axa.set_ylim(0, ytop * 1.35)
axa.set_xlabel("Occupancy (species with the HOG)")
axa.set_ylabel("HOGs")
axa.set_title("HOGs tested at each occupancy", loc="left")
axa.grid(axis="x", visible=False)

bins = [(30, 49), (50, 69), (70, 88), (89, max_occ_all)]
blabels = [f"{a}\u2013{b}" for a, b in bins]
for test, grp in TESTS:
    d = long[(long["test"] == test) & (long["group"] == grp)]
    rates = []
    for a, b in bins:
        sub = d[(d["occ"] >= a) & (d["occ"] <= b)]
        rates.append((sub["p"] <= TAIL_ALPHA).mean() if len(sub) else np.nan)
    axb.plot(blabels, rates, color=COL[grp], linestyle="-" if test == "loss" else "--",
             marker="o" if test == "loss" else "s", markersize=6, linewidth=1.8,
             markeredgecolor="white", markeredgewidth=1.2)
axb.axhline(TAIL_ALPHA, color=INK, linestyle=":", linewidth=1)
axb.text(-0.08, TAIL_ALPHA + 0.0015, "2.5% (null)", ha="left", va="bottom", fontsize=8.5, color=MUTED)
axb.set_xlabel("Occupancy bin")
axb.set_ylabel("Fraction of tested HOGs with p \u2264 0.025")
axb.set_title("Hit rate by occupancy", loc="left")
axb.set_ylim(bottom=0)
axb.legend(handles=[
    Line2D([], [], color=COL["fg"], lw=2, label="orb-weavers"),
    Line2D([], [], color=COL["bg"], lw=2, label="non-orb-weavers"),
    Line2D([], [], color=INK, lw=1.8, marker="o", markeredgecolor="white", label="loss"),
    Line2D([], [], color=INK, lw=1.8, ls="--", marker="s", markeredgecolor="white", label="duplication"),
], frameon=False, fontsize=8.5, loc="best")
fig.tight_layout()
save(fig, "ort_occupancy.png")
plt.show()

# Loss p-values for HOGs above the loss occupancy bound: these appear in fltrd_hits.csv only because
# they are duplication hits, so any loss list built from fltrd_hits must re-apply the occupancy filter.
fh = pd.read_csv(os.path.join(ort_dir, "fltrd_hits.csv"), index_col="HOG")
lp = [c for c in fh.columns if c.startswith("P-value loss")]
leak = fh[(fh["Occupancy"] > MAX_OCC_LOSS) & (fh[lp] <= TAIL_ALPHA).any(axis=1)]
print(f"{len(leak)} HOGs in fltrd_hits.csv have occupancy > {MAX_OCC_LOSS} and a loss p-value <= {TAIL_ALPHA} "
      f"(outside the loss universe)")

## 4. LOR vs. p-value, with the permulation thresholds

Each point is a HOG. The y axis is -log10(p), and p = 0 is drawn at the floor with a triangle.
The solid vertical line is the permulation-derived cutoff (average permulated mean ± 1.96 sd, i.e. two-tailed). Each direction is called at
p ≤ 0.025, which corresponds to the same z = 1.96. Colored points pass both p ≤ 0.025 and the LOR cutoff. Open points pass the p-value only.

In [ ]:
Z2 = norm.ppf(1 - ALPHA / 2)
floor = 0.5 / REPS

fig, axs = plt.subplots(2, 2, figsize=(10.5, 8), sharey=True)
for ax, (test, grp) in zip(axs.flat, TESTS):
    d = long[(long["test"] == test) & (long["group"] == grp)].copy()
    m, s = perm_stats[test]
    sign = 1 if grp == "fg" else -1
    cut2 = m + sign * Z2 * s
    d["y"] = -np.log10(d["p"].clip(lower=floor))
    p_ok = d["p"] <= TAIL_ALPHA
    beyond2 = d["lor"] > cut2 if grp == "fg" else d["lor"] < cut2
    both, p_only, neither = p_ok & beyond2, p_ok & ~beyond2, ~p_ok
    ax.scatter(d.loc[neither, "lor"], d.loc[neither, "y"], s=5, color="#c4c4c4", alpha=0.5,
               linewidths=0, rasterized=True)
    ax.scatter(d.loc[p_only, "lor"], d.loc[p_only, "y"], s=22, facecolors="none",
               edgecolors=COL[grp], linewidths=0.9, alpha=0.9)
    ax.scatter(d.loc[both, "lor"], d.loc[both, "y"], s=22, color=COL[grp], edgecolors="white",
               linewidths=0.4)
    zero = d["p"] == 0
    ax.scatter(d.loc[zero, "lor"], d.loc[zero, "y"], s=60, marker="^", color=INK, edgecolors="white",
               linewidths=0.6, zorder=5)
    ax.axhline(-np.log10(TAIL_ALPHA), color=INK, linewidth=0.8, linestyle=":")
    ax.axvline(cut2, color=INK, linewidth=1.3)
    ax.set_title(f"{WORD[test].capitalize()} more likely in {NAME[grp]}", loc="left", pad=20)
    ax.text(0, 1.015,
            f"p \u2264 0.025: {int(p_ok.sum())}  |  also beyond LOR cutoff: {int(both.sum())}",
            transform=ax.transAxes, ha="left", va="bottom", fontsize=8.5, color=MUTED)
    ax.set_xlabel("Log odds ratio")
for ax in axs[:, 0]:
    ax.set_ylabel("-log10(permulation p)")
fig.legend(handles=[
    Line2D([], [], marker="o", ls="", color="#c4c4c4", label="p > 0.025"),
    Line2D([], [], marker="o", ls="", mfc="none", mec=INK, label="p \u2264 0.025, inside LOR cutoff"),
    Line2D([], [], marker="o", ls="", color=INK, mec="white", label="p \u2264 0.025 and beyond LOR cutoff"),
    Line2D([], [], marker="^", ls="", color=INK, mec="white", label="p = 0 (drawn at floor)"),
    Line2D([], [], color=INK, lw=1.3, label=f"LOR cutoff (z={Z2:.2f}, two-tailed)"),
], frameon=False, fontsize=8.5, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.01))
fig.suptitle("LOR vs. permulation p-value and permulation-derived LOR thresholds", x=0.01, ha="left",
             fontsize=12)
fig.tight_layout(rect=(0, 0.07, 1, 0.97))
save(fig, "ort_lor_vs_p.png")
plt.show()

## 5. How close do the strongest HOGs sit to the p-value floor?

An empirical permulation p-value cannot go below 1/reps, and BH needs a minimum number of HOGs sitting *at* that floor
before any of them can pass (both tails of a test are corrected together, so the number of tests is twice the number of HOGs). The table gives that number for each test and compares the number of extreme HOGs observed with
what a uniform null would produce. Re-run this section after the larger permulation run, since the floor drops with more permulations.

In [ ]:
floor_p = 1 / REPS
rows = []
for (test, grp), d in long.groupby(["test", "group"], sort=False):
    m = len(d)
    rows.append({
        "test": f"{WORD[test]}, {NAME[grp]}",
        "HOGs tested": m,
        "p = 0": int((d["p"] == 0).sum()),
        "p <= 1e-4": int((d["p"] <= 1e-4).sum()),
        "p <= 5e-4": int((d["p"] <= 5e-4).sum()),
        "expected p <= 5e-4 if uniform": round(m * 5e-4, 1),
        f"HOGs needed at the floor ({floor_p:g}) to pass BH": int(np.ceil(floor_p * 2 * m / ALPHA)),
        "Bonferroni threshold": f"{ALPHA / (2 * m):.1e}",
    })
pd.DataFrame(rows).set_index("test")